# Figure 1 — A time-resolved map of regulators and gene programs during arterial endothelial differentiation

Second-pass implementation of code panels b–d. Panel a remains an editable source asset.


In [ ]:
import matplotlib.pyplot as plt
from src.cc_perturbseq.figure_notebook import get_path, initialize_figure, load_figure_manifest, panel_directory, source_asset_status, write_receipt
from src.cc_perturbseq.program_panels import *

def export(fig, panel, stem):
    out = panel_directory(ctx, panel)
    result = {}
    for suffix, kwargs in [("pdf", {}), ("svg", {}), ("png", {"dpi": 300})]:
        path = out / f"{stem}.{suffix}"
        fig.savefig(path, **kwargs)
        result[suffix] = str(path)
    return result

FIGURE = load_figure_manifest("Figure_01")
ctx = initialize_figure(FIGURE)
ctx["input_audit"]


## Panel a — experimental framework schematic


In [ ]:
source_asset_status(ctx, "a", "assets.figure1a_schematic")


## Panel b — genome-wide CRISPRi screen


In [ ]:
mageck_path = get_path(ctx["paths"], "screen.genomewide_mageck_pos_vs_neg")
utest_path = get_path(ctx["paths"], "screen.genomewide_utest_pos_vs_neg")
screen = load_genomewide_screen(mageck_path, utest_path)
fig, volcano_source = plot_genomewide_volcano(screen)
outputs = export(fig, "b", "Figure_01b_genomewide_screen_volcano")
tables = save_source_tables({"screen_points": volcano_source}, panel_directory(ctx, "b"), "Figure_01b")
write_receipt(ctx, "b", status="IMPLEMENTED_NOT_YET_NUMERICALLY_VALIDATED", inputs={"mageck": str(mageck_path), "utest": str(utest_path)}, outputs={**outputs, **tables}, parameters={"fdr_threshold": 0.05, "directional_fdr": True})
plt.show()


## Panel c — selected guide-level Day 3 EC fractions


In [ ]:
ec_path = get_path(ctx["paths"], "ccperturbseq.ec_fraction_table")
ec_table = standardize_ec_fraction_table(ec_path)
selected = [gene for genes in FIGURE1_TARGET_GROUPS.values() for gene in genes]
fig, ec_source = plot_guide_ec_fraction(ec_table, targets=selected, target_groups=FIGURE1_TARGET_GROUPS)
outputs = export(fig, "c", "Figure_01c_selected_ec_fraction")
tables = save_source_tables({"guide_ec_fraction": ec_source}, panel_directory(ctx, "c"), "Figure_01c")
write_receipt(ctx, "c", status="IMPLEMENTED_NOT_YET_NUMERICALLY_VALIDATED", inputs={"ec_fraction_table": str(ec_path)}, outputs={**outputs, **tables}, parameters={"targets": selected, "point_unit": "guide"})
plt.show()


## Panel d — integrated program atlas


In [ ]:
adata_path = get_path(ctx["paths"], "ccperturbseq.integrated_anndata")
usage_path = get_path(ctx["paths"], "ccperturbseq.cnmf_usage")
annotation_path = get_path(ctx["paths"], "ccperturbseq.program_annotations", required=False)
adata, usage = load_integrated_anndata(adata_path, usage_path)
labels, _ = load_program_annotations(annotation_path)
PROGRAMS_BY_DAY = {"D0": [7,22,29], "D1": [28,13,30,48,33], "D2": [15,23,14,17,19], "D3": [2,3,8,11,24,27,34,35]}
fig, atlas_source = plot_program_atlas(adata, usage, programs_by_day=PROGRAMS_BY_DAY, program_labels=labels)
outputs = export(fig, "d", "Figure_01d_integrated_program_atlas")
tables = save_source_tables({"cell_program_assignment": atlas_source}, panel_directory(ctx, "d"), "Figure_01d")
write_receipt(ctx, "d", status="IMPLEMENTED_NOT_YET_NUMERICALLY_VALIDATED", inputs={"anndata": str(adata_path), "usage": str(usage_path), "annotations": str(annotation_path)}, outputs={**outputs, **tables}, parameters={"programs_by_day": PROGRAMS_BY_DAY})
plt.show()
